In [1]:
import ROOT
ROOT.gStyle.SetOptStat(0)

## Part 2. Yield of $B_s^0 \to \mu^+ \mu^-$ 

2.f Let's check if your prediction on number of $B_s^0 \to J/\psi \mu^+ \mu^-$ is correct. 


You have four input files:

+ bs2mumu_toy.root contains the toy simulation of the $B_s^0 \to \mu^+ \mu^-$ mass range.
+ peaking_bkg_toy.root contains the toy simulation of the peaking backgrounds, such as $B\to hh$.
+ semi_bkg_toy.root containe the toy simulation of the semileptonic backgrounds, such as $B \to h \mu \nu$. 
+ full_toy.root contains the toy simulation of the dimuon spectra near the $B_s^0$ mass range.

Use the first three to find out the shapes for the full model. Which shapes out of three you would fix and which not? 

In [2]:
#Open the files 
filein_sig = ROOT.TFile.Open("bs2mumu_toy.root")
signal_shape = filein_sig.Get("signal_tree")
filein_peak = ROOT.TFile.Open("peaking_bkg_toy.root")
peaking_shape =  filein_peak.Get("peak_tree")
fileinsemi = ROOT.TFile.Open("semi_bkg_toy.root")
semi_bkg_shape = fileinsemi.Get("semi_tree")
fileinfulltoy = ROOT.TFile.Open("full_toy.root")
full_toy = fileinfulltoy.Get("Events")

In [5]:
nbins = 80
xmin  = 4.9
xmax  = 6.0

h2_full  = ROOT.TH1D("h2_full","",nbins,xmin,xmax)
h2_sig   = ROOT.TH1D("h2_sig",r";m(\mu^+\mu^-) [GeV];Normalized entries",nbins,xmin,xmax)
h2_peak  = ROOT.TH1D("h2_peak","",nbins,xmin,xmax)
h2_semi = ROOT.TH1D("h2_semi","",nbins,xmin,xmax)

full_toy.Draw("mass>>h2_full","","goff")
signal_shape.Draw("mass>>h2_sig","","goff")
peaking_shape.Draw("mass>>h2_peak","","goff")
semi_bkg_shape.Draw("mass>>h2_semi","","goff")
    
h2_full.SetLineColor(ROOT.kBlack)
h2_full.SetLineWidth(2)
h2_full.Scale(1.0/h2_full.Integral())

h2_sig.SetLineColor(ROOT.kBlue)
h2_sig.SetLineWidth(2)
h2_sig.Scale(1.0/h2_sig.Integral())

h2_peak.SetLineColor(ROOT.kRed)
h2_peak.SetLineWidth(2)
h2_peak.Scale(1.0/h2_peak.Integral())

h2_semi.SetLineColor(ROOT.kMagenta)
h2_semi.SetLineWidth(2)
h2_semi.Scale(1.0/h2_semi.Integral())

cbs = ROOT.TCanvas("cbs","Bs2MuMu shapes",900,700)
h2_sig.Draw("")
h2_full.Draw("same")
h2_peak.Draw("same")
h2_semi.Draw("same")

leg = ROOT.TLegend(0.65,0.70,0.88,0.88)
leg.AddEntry(h2_full,"Full toy sample","l")
leg.AddEntry(h2_sig,r"B_{s}^{0} #rightarrow \mu^{+} \mu^{-} signal","l")
leg.AddEntry(h2_peak,"Peaking background","l")
leg.AddEntry(h2_semi,"Semilept. background","l")
leg.Draw()
cbs.Modified()
cbs.Update()
cbs.SaveAs("bs_shapes_histogram.pdf")


Warning in <TNetXNGFile::Append>: Replacing existing TH1: h2_full (Potential memory leak).
Warning in <TNetXNGFile::Append>: Replacing existing TH1: h2_sig (Potential memory leak).
Warning in <TNetXNGFile::Append>: Replacing existing TH1: h2_peak (Potential memory leak).
Warning in <TNetXNGFile::Append>: Replacing existing TH1: h2_semi (Potential memory leak).
Warning in <TCanvas::Constructor>: Deleting canvas with same name: cbs
Info in <TCanvas::Print>: pdf file bs_shapes_histogram.pdf has been created


First we find all the input shapes. Both the semileptonic and peaking backgrounds are described empirically with a one-dimensional kernel estimation p.d.f RooKeysPdf 

In [7]:
massBs = ROOT.RooRealVar("mass", "m(#mu#mu)", 4.9, 6.0, "GeV")

dataset4 = ROOT.RooDataSet("data4", "", ROOT.RooArgList(massBs), ROOT.RooFit.Import(peaking_shape))

peak_pdf = ROOT.RooKeysPdf(
    "peak_pdf", "Peaking background keys pdf",
    massBs, dataset4, ROOT.RooKeysPdf.NoMirror, 2.0
)
N_peak = ROOT.RooRealVar("N_peak", "N_peak", 10000, 0, 50000)

c4 = ROOT.TCanvas("c4","fit",800,600)
frame = massBs.frame()
dataset4.plotOn(frame)
peak_pdf.plotOn(frame)
frame.Draw()
c4.Update()
c4.SaveAs("bs_peaking_bkg.pdf")

[#1] INFO:DataHandling -- RooAbsReal::attachToTree(mass) TTree Float_t branch mass will be converted to double precision.


Info in <TCanvas::Print>: pdf file bs_peaking_bkg.pdf has been created


Looking at the peaking background shape, now modelled with kernel density function, which alternative modelling shapes would you propose? 

In [8]:
dataset5 = ROOT.RooDataSet("data5", "", ROOT.RooArgList(massBs), ROOT.RooFit.Import(semi_bkg_shape))

semi_pdf = ROOT.RooKeysPdf(
    "semi_pdf", "Semileptonic keys pdf",
    massBs, dataset5, ROOT.RooKeysPdf.NoMirror, 2.0
)
N_semi = ROOT.RooRealVar("N_semi", "N_semi", 10000., 0., 50000.)

c5 = ROOT.TCanvas("c5","fit",800,600)
frame = massBs.frame()
dataset5.plotOn(frame)
semi_pdf.plotOn(frame)
frame.Draw()
c5.Update()
c5.Show()

[#1] INFO:DataHandling -- RooAbsReal::attachToTree(mass) TTree Float_t branch mass will be converted to double precision.


In [9]:
dataset6 = ROOT.RooDataSet("data6", "", ROOT.RooArgList(massBs), ROOT.RooFit.Import(signal_shape))




#set up values for fitting
mean_init = 0
mean_min =0
mean_max =0
sigma_init = 0
sigma_min =0
sigma_max =0
alpha_init = 0
sigma_min =0
sigma_max =0

meanBs   = ROOT.RooRealVar("meanBs",   "signal mean",   mean_init, mean_min, mean_max)
sigmaBs1 = ROOT.RooRealVar("sigmaBs1", "CB sigma",        sigma_init, sigma_min, sigma_max)
alphaBs  = ROOT.RooRealVar("alphaBs",  "CB alpha",        alpha_init, alpha_min, alpha_max)
nBs      = ROOT.RooRealVar("nBs",      "CB n",            n_init, n_min, n_max)

sigBs_pdf = ROOT.RooCBShape("cbBs", "Crystal Ball", massBs, meanBs, sigmaBs1, alphaBs, nBs)

N_bs = ROOT.RooRealVar("N_bs", "N_bs", 10000, 0., 60000)



fit_result = sigBs_pdf.fitTo(dataset6, 
                                 ROOT.RooFit.Minimizer("Minuit2"),
                                 ROOT.RooFit.Optimize(True), #optimize the treatment of constants in logL
                                 ROOT.RooFit.Offset(True), #set initial logL to 0
                                 ROOT.RooFit.Strategy(2), #internal code for the most precise minimization strategy of minuit2
                                 ROOT.RooFit.Hesse(True), #use Hesse to compute the uncertainty
                                 ROOT.RooFit.Save(True))

fit_result.Print("v")
c6 = ROOT.TCanvas("c5","fit",800,600)
frame = massBs.frame()
dataset6.plotOn(frame)
sigBs_pdf.plotOn(frame)
frame.Draw()
c6.Update()
c6.Show()

[#1] INFO:DataHandling -- RooAbsReal::attachToTree(mass) TTree Float_t branch mass will be converted to double precision.
[#1] INFO:Fitting -- RooAbsPdf::fitTo(cbBs_over_cbBs_Int[mass]) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- Creation of NLL object took 1.42169 ms
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_cbBs_over_cbBs_Int[mass]_data6) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: activating const optimization
[#1] INFO:Minimization -- [fitFCN] No discrete parameters, performing continuous minimization only
Minuit2Minimizer: Minimize with max-calls 2000 convergence for edm < 1 strategy 2
Minuit2Minimizer : Valid minimum - status = 0
FVAL  = -25336.6792495000846
Edm   = 5.56884324929074061e-05
Nfcn  = 228
alphaBs	  = 1.48913	 +/-  0.0268334	(limited)
meanBs	  = 5.36696	 +/-  0.0002836	(limited)
nBs	  = 2.96066	 +/-  0.15215	(limited)
sigmaBs1	

Info in <Minuit2>: MnSeedGenerator Computing seed using NumericalGradient calculator
Info in <Minuit2>: MnSeedGenerator Evaluated function and gradient in 35.4235 ms
Info in <Minuit2>: MnSeedGenerator Initial state: FCN =                 0 Edm =       69005.83441 NCalls =     17
Warning in <Minuit2>: MnPosDef Matrix forced pos-def by adding to diagonal 2.57665
Info in <Minuit2>: MnHesse Done after 44.5934 ms
Info in <Minuit2>: MnSeedGenerator run Hesse - Initial seeding state: 
  Minimum value : 0
  Edm           : 1356880.964
  Internal parameters:	[    -0.4761190609   -0.03429243504    -0.7765890262    -0.6082455789]	
  Internal gradient  :	[      25770.34604      -11559.2631      13787.05888      -101616.364]	
  Internal covariance matrix:
[[   0.0029134952   0.0014822188   -0.016612344  -0.0004538912]
 [   0.0014822188   0.0007986277  -0.0085021732  -0.0002342452]
 [   -0.016612344  -0.0085021732    0.095075919   0.0026014269]
 [  -0.0004538912  -0.0002342452   0.0026014269   7.298

In the real analysis, you will never directly look at the interesting data while developing the analysis methodology. 
This is done to prevent introducing a bias in the methodology. 
Imagine you want to discover new physics and you introduce a mass-shaping effect that creates a fake "2-sigma" peak, hopefully uncontiously. 
Thus, typically blinding is introduced. This means an analyser does not look directly at the region of interest. 
Our region of interest is a $B_s^0 \to \mu^+ \mu^-$ peak. 
Define a blinding region based on the bs2mumu_toy.root sample. 

In [11]:
#Add shape constraints 
alphaBs.setConstant(True)
nBs.setConstant(True)

dataset7 = ROOT.RooDataSet("data6", "", ROOT.RooArgList(massBs), ROOT.RooFit.Import(full_toy))

gamma = ROOT.RooRealVar("gamma", "gamma", -0.0001, -0.2, 0.1)
comb_pdf = ROOT.RooExponential("comb_pdf", "Combinatorial", massBs, gamma)

mean_b0 = ROOT.RooRealVar("mean_b0", "", 5.279) #fix to the b0 mass from pdg
mean_b0.setConstant(True)
sigma_b0 = ROOT.RooRealVar("sigma_b0", "", sigmaBs1.getVal()) #fix to the b0 mass from pdg
sigma_b0.setConstant(True)
sigB0_pdf = ROOT.RooGaussian("pdf_b0", "", massBs, mean_b0, sigma_b0)


N_bs2 = ROOT.RooRealVar("N_bs2", "N_bs", 0.5*dataset7.numEntries(), 0., 1.2*dataset7.numEntries())
N_comb2 = ROOT.RooRealVar("N_comb2", "N_comb", 6426.8, 0., 1.2*dataset7.numEntries())
N_semi2 = ROOT.RooRealVar("N_semi2", "N_semi", 10000., 0., 1.2*dataset7.numEntries())
N_peak2 = ROOT.RooRealVar("N_peak2", "N_peak2", 10000., 0., 1.2*dataset7.numEntries())
N_b0 = ROOT.RooRealVar("N_b0", "N_b0", 0., 50.)



modelBs = ROOT.RooAddPdf("modelBs", "", ROOT.RooArgList(sigBs_pdf, sigB0_pdf, peak_pdf, semi_pdf, comb_pdf), ROOT.RooArgList(N_bs2, N_b0, N_peak2, N_semi2, N_comb2))
fit_result = modelBs.fitTo(dataset7, 
                                 ROOT.RooFit.Minimizer("Minuit2"),
                                 ROOT.RooFit.Extended(True), #it is extended fit
                                 ROOT.RooFit.Optimize(True), #optimize the treatment of constants in logL
                                 ROOT.RooFit.Offset(True), #set initial logL to 0
                                 ROOT.RooFit.Strategy(2), #internal code for the most precise minimization strategy of minuit2
                                 ROOT.RooFit.Hesse(True), #use Hesse to compute the uncertainty
                                 ROOT.RooFit.Save(True))


fit_result.Print("v")

c7 = ROOT.TCanvas("c7","fit",800,600)
frame = massBs.frame()
dataset7.plotOn(frame)
modelBs.plotOn(frame)
modelBs.plotOn(frame, ROOT.RooFit.Components("cbBs"), ROOT.RooFit.LineColor(ROOT.kGreen))

frame.Draw()
c7.Update()
c7.SaveAs("bs_fit.png")
c7.Draw()


[#1] INFO:DataHandling -- RooAbsReal::attachToTree(mass) TTree Float_t branch mass will be converted to double precision.
[#0] WARNING:InputArguments -- The parameter 'sigma_b0' with range [-inf, inf] of the RooGaussian 'pdf_b0' exceeds the safe range of (0, inf). Advise to limit its range.
[#1] INFO:Fitting -- RooAbsPdf::fitTo(modelBs) fixing normalization set for coefficient determination to observables in data
[#1] INFO:Fitting -- Creation of NLL object took 525.206 μs
[#1] INFO:Fitting -- RooAddition::defaultErrorLevel(nll_modelBs_data6) Summation contains a RooNLLVar, using its error level
[#1] INFO:Minimization -- RooAbsMinimizerFcn::setOptimizeConst: activating const optimization
[#1] INFO:Minimization -- [fitFCN] No discrete parameters, performing continuous minimization only
Minuit2Minimizer: Minimize with max-calls 4000 convergence for edm < 1 strategy 2
Minuit2Minimizer : Valid minimum - status = 0
FVAL  = -13215.6880287140084
Edm   = 5.96533550323701271e-05
Nfcn  = 815
N_b0

Info in <Minuit2>: MnSeedGenerator Computing seed using NumericalGradient calculator
Info in <Minuit2>: MnSeedGenerator Evaluated function and gradient in 6.92546 ms
Info in <Minuit2>: MnSeedGenerator Initial state: FCN =                 0 Edm =        567033.979 NCalls =     31
Info in <Minuit2>: NegativeG2LineSearch Doing a NegativeG2LineSearch since one of the G2 component is negative
Info in <Minuit2>: NegativeG2LineSearch Done after 36.472 ms
Info in <Minuit2>: MnSeedGenerator Negative G2 found - new state: 
  Minimum value : -10378.49742
  Edm           : 6696.757164
  Internal parameters:	[                0    -0.1674480792     0.5182603948    -0.4930133002     -1.513957958      0.339129891      1.567395869     0.1473768049]	
  Internal gradient  :	[      14.10463412      2974.614641      532.1001772      2836.415297      27.33269296     -2.433350665    -0.1970494731     -344.3911356]	
  Internal covariance matrix:
[[      20.692736              0              0              0  

2. g Using the obtained value of the $B_s^0 \to \mu^+ \mu^-$ events, $B^+ \to  J/\psi K^+$ events from the previous question, the efficiencies $\varepsilon_{B^+\to J/\psi K^+} = 0.001$ and $\varepsilon_{B^+\to\mu^+\mu^-} = 0.04$, and $f_s/f_u = 0.256$, compute the branching ratio you get. Propagate the statistical uncertianties from the number of events. What are the systematic effects you expect? 

2. h $p_T$ dependence in the $f_s/f_u$ has been observed by LHCb in the https://journals.aps.org/prd/abstract/10.1103/PhysRevD.100.031102. Can you use LHCb result in this CMS-like measurement? If yes, how do you propagate systematics on it? If not, how would you measure it within this analysis? 

## Part 3 : search for $B^0 \to \mu^+ \mu^-$

The same dataset contains a small amount of $B^0 \to \mu^+ \mu^-$. 
In Standard Model BR($B^0 \to \mu^+ \mu^-$) is $\mathcal{O}(10^{-10})$. 
Why is it suppressed compared to the $B^0_s \to \mu^+ \mu^-$? 

Let's search for $B^0 \to \mu^+ \mu^-$in the sample. 
Complete this code. How does UL depend on the efficiency?


In [2]:
win_lo = mean_b0.getVal() - 2.0 * sigma_b0.getVal()
win_hi = mean_b0.getVal() + 2.0 * sigma_b0.getVal()

massBs.setRange("b0_window", win_lo, win_hi)

print("\n--- B0 signal window ---")
print(f"B0 mean      = {mean_b0.getVal():.3f} MeV")
print(f"sigma_eff    = {sigma_b0.getVal():.3f} MeV")
print(f"window       = [{win_lo:.3f}, {win_hi:.3f}] MeV")

x_obs = N_b0.getVal()

print(f"Observed events in B0 window = {x_obs}")


frac_comb = comb_pdf.createIntegral(
    ROOT.RooArgSet(massBs),
    ROOT.RooFit.NormSet(ROOT.RooArgSet(massBs)),
    ROOT.RooFit.Range("b0_window")
).getVal()

frac_peak = peak_pdf.createIntegral(
    ROOT.RooArgSet(massBs),
    ROOT.RooFit.NormSet(ROOT.RooArgSet(massBs)),
    ROOT.RooFit.Range("b0_window")
).getVal()

frac_semi = semi_pdf.createIntegral(
    ROOT.RooArgSet(massBs),
    ROOT.RooFit.NormSet(ROOT.RooArgSet(massBs)),
    ROOT.RooFit.Range("b0_window")
).getVal()

frac_bs = sigBs_pdf.createIntegral(
    ROOT.RooArgSet(massBs),
    ROOT.RooFit.NormSet(ROOT.RooArgSet(massBs)),
    ROOT.RooFit.Range("b0_window")
).getVal()

b_comb = #amoutn of comb_bkg 
b_peak = #amoutn of peak 
b_semi = #amoutn of semi 
b_bs   = #amoutn of Bs 

b_exp = #expected background
print(f"Expected events in B0 window = {b_exp}")

#Propagate error treating fraction as constant 
err_comb = N_comb2.getError() * frac_comb
err_peak = N_peak2.getError() * frac_peak
err_semi = N_semi2.getError() * frac_semi
err_bs   = N_bs2.getError()   * frac_bs

db_exp =  #add in quadratures 

print("\n--- Background estimate in B0 window ---")
print(f"comb = {b_comb:.3f} +/- {err_comb:.3f}")
print(f"peak = {b_peak:.3f} +/- {err_peak:.3f}")
print(f"semi = {b_semi:.3f} +/- {err_semi:.3f}")
print(f"Bs   = {b_bs:.3f} +/- {err_bs:.3f}")
print(f"Total background = {b_exp:.3f} +/- {db_exp:.3f}")


rolke = ROOT.TRolke()
rolke.SetCL(0.95)


#How does the UL depend on the efficiency? 
eff_count =1.0 #pseudo eff
deff_count = 0. #pseudo eff err


rolke.SetGaussBkgGaussEff(int(x_exp), b_exp, db_exp, eff_count, deff_count)

s_ul = rolke.GetUpperLimit()

print("\n--- TRolke expected result for B0 -> mu mu ---")
print(f"95% CL lower limit on B0 signal yield = {s_ul:.4f}")

rolke.SetGaussBkgGaussEff(0, b_exp, db_exp, eff_count, deff_count)

s_ul = rolke.GetUpperLimit()

print("\n--- TRolke result for B0 -> mu mu ---")
print(f"95% CL lower limit on B0 signal yield = {s_ul:.4f}")




NameError: name 'mean_b0' is not defined